 Make sure you have the datasets, transformer and torch libraries

In [3]:
# install the necessary libraries
%pip install datasets
%pip install transformer
%pip install torch


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement transformer (from versions: none)

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for transformer


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


 Now we import the libraries

In [4]:
# import the libraries and functions we need
import transformers, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import datasets as ds



c:\Users\jaxson\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

#initialize one variables including the name of the model we want to use
#the tokenizer we want to use and the actual model
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


c:\Users\jaxson\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jaxson\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\jaxson\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\toke

 Make the Model run on the GPU

In [ ]:

#set the device to cuda if the GPU is available, else set it to cpu
#then tell the model to use that device
if torch.cuda.is_available():
    device = "cuda"
else: device = "cpu"

print(device)
model = model.to(device)


AssertionError: Torch not compiled with CUDA enabled

In [6]:
# load the training and testing jsonl data sets  and turn them into data sets from datasetDict's
dataset = ds.load_dataset("json", data_files="shakespearTrain.jsonl",encoding='utf-8')
testDataset = ds.load_dataset("json", data_files="shakespearTest.jsonl",encoding='utf-8')


if isinstance(dataset, ds.DatasetDict):
    # Concatenate all splits into a single Dataset
    trainDataset = ds.concatenate_datasets([dataset[split] for split in dataset])
else:
    raise ValueError("Loaded data is not a DatasetDict")

if isinstance(testDataset, ds.DatasetDict):
    # Concatenate all splits into a single Dataset
    testDataset = ds.concatenate_datasets([testDataset[split] for split in testDataset])
else:
    raise ValueError("Loaded data is not a DatasetDict")
#


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
#preproceesss the data from the jsonl file
def preprocess(dataset):
    #seperate the different types of data we are giving it,
    #training data that is seperated into prompts and responses
    inputs = dataset["prompt"]
    responses = dataset["response"]

    # Convert inputs and responses to lists of strings
    inputs = [str(i) for i in inputs]
    responses = [str(r) for r in responses]


    # Tokenize the articles (inputs) with padding and truncation to a max length of 512
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True, return_tensors="pt")

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(responses, max_length=128, padding="max_length", truncation=True, return_tensors="pt")

    model_inputs["labels"] = labels["input_ids"]

    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

    return model_inputs

In [11]:
#map the data to a useable format for the trainer
tokenized_train_dataset = trainDataset.map(preprocess, batched=True)
print("done")

tokenized_eval_dataset = testDataset.map(preprocess, batched=True)


Map:   0%|          | 0/17866 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4118: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4118: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4118: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your 

done


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4118: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/560 [00:00<?, ? examples/s]

In [17]:
# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir='./results',              # Directory to save the model checkpoints
    eval_strategy="epoch",         # Evaluate the model at the end of every epoch
    learning_rate=2e-5,                  # Learning rate for the optimizer
    per_device_train_batch_size=8,       # Batch size for training
    per_device_eval_batch_size=8,        # Batch size for evaluation
    weight_decay=0.01,                   # Regularization to prevent overfitting
    save_total_limit=3,                  # Only keep the last 3 checkpoints
    num_train_epochs=4,                  # Number of training epochs
    predict_with_generate=True,          # Enable text generation during evaluation
    logging_dir="./logs",  # Directory for storing training logs
    report_to="none"
)


In [18]:
# Create the trainer object
trainer = Seq2SeqTrainer(
    model=model,                         # The model to be trained
    args=training_args,                  # The training arguments defined earlier
    train_dataset=tokenized_train_dataset,  # The tokenized training dataset
    eval_dataset=tokenized_eval_dataset,    # The tokenized evaluation dataset
    tokenizer=tokenizer                  # The tokenizer to handle input and output
)


/tmp/ipython-input-2561201102.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [19]:
#TRAINN!!!! and wait :(
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.231900,0.215349
2,0.150700,0.140630
3,0.120900,0.112639
4,0.109900,0.104375


TrainOutput(global_step=8936, training_loss=0.6000412613980559, metrics={'train_runtime': 3743.0578, 'train_samples_per_second': 19.092, 'train_steps_per_second': 2.387, 'total_flos': 1.3284479028166656e+16, 'train_loss': 0.6000412613980559, 'epoch': 4.0})

In [ ]:
# Evaluate the model on the testing dataset
metrics = trainer.evaluate()

# Print the evaluation metrics
print(metrics)


{'eval_loss': 0.10405918210744858, 'eval_runtime': 10.072, 'eval_samples_per_second': 55.6, 'eval_steps_per_second': 6.95, 'epoch': 4.0}


In [1]:
# save the model
REPO_NAME = "JaxsonYorke/EnglishToShakespearean"

# save model and tokenizer
model.save_pretrained(REPO_NAME)
tokenizer.save_pretrained(REPO_NAME)


NameError: name 'model' is not defined

In [24]:
def askQuestion(text):
  # Tokenize the input text and move it to the correct device
  inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)

  # Load the model from the saved REPO_NAME
  model = AutoModelForSeq2SeqLM.from_pretrained(REPO_NAME).to(device)

  # Generate the summary using the fine-tuned model
  response_ids = model.generate(inputs["input_ids"], max_length=2048, num_beams=4, early_stopping=True)

  # Decode the generated summary back into text and return it
  return tokenizer.decode(response_ids[0], skip_special_tokens=True)

In [28]:
print(askQuestion("Take the winding path to reach the lake and open your book to the first page."))

Taketh the winding path to reacheth the lake and open thy booketh to the first page
